 # 1. Import Library

In [ ]:
!pip install nibabel
!pip install monai 

In [ ]:
import os
import shutil
from pathlib import Path
import zipfile

from tqdm import tqdm
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from dataclasses import dataclass
from typing import List, Optional, Tuple
import tarfile
import random
from numpy import mean, std
from sklearn.model_selection import train_test_split

import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score  

from sklearn.model_selection import train_test_split



 # 2. Data

 ## 2.1 Data Extraction

In [ ]:

import kagglehub

path = kagglehub.dataset_download("dschettler8845/brats-2021-task1")
print("Path to dataset files:", path)

brats_dir = "./BraTS2021_Training_Data2/BraTS2021_Training_Data2"

for tar_file in os.listdir(path):
	if tar_file.endswith(".tar"):
		tar_path = os.path.join(path, tar_file)
		with tarfile.open(tar_path, 'r') as tar_ref:
			tar_ref.extractall(brats_dir)




In [ ]:
def is_case_complete(case_dir):
	nii_files = list(Path(case_dir).glob("*.nii*"))
	return len(nii_files) >= 4  

valid_cases = []

for case in Path(brats_dir).iterdir():
	if case.is_dir() and is_case_complete(case):
		valid_cases.append(case)


 ## 2.2 Volume Calculation Function

In [ ]:
def whole_tumor_volume_mm3(seg_path):
	img = nib.load(seg_path)
	seg = img.get_fdata()
	voxel_spacing = img.header.get_zooms()[:3]
	voxel_volume = np.prod(voxel_spacing)
	wt_mask = np.isin(seg, [1, 2, 4])
	volume_mm3 = np.sum(wt_mask) * voxel_volume
	return volume_mm3


 ## 2.3 nnU-Net Installation and Setup

In [ ]:
# ------
!git clone https://github.com/rixez/Brats21_KAIST_MRI_Lab.git
repo_path = r"./Brats21_KAIST_MRI_Lab"

#------
req_file = os.path.join(repo_path, "requirements_v2.txt")
with open(req_file, "r") as f:
	lines = f.readlines()
lines = [line for line in lines if line.strip() != "sklearn==0.0"]
with open(req_file, "w") as f:
	f.writelines(lines)

#------
setup_file = os.path.join(repo_path, "setup.py")
with open(setup_file, "r") as f:
	lines = f.readlines()
new_lines = []
skip_next_indent_check = False
for line in lines:
	stripped = line.strip()
	if skip_next_indent_check:
		skip_next_indent_check = False
		continue

	if '"sklearn",' in stripped:
		continue

	new_lines.append(line)
with open(setup_file, "w") as f:
	f.writelines(new_lines)

#---------
file_path_train = r"./Brats21_KAIST_MRI_Lab/nnunet/training/model_restore.py"
with open(file_path_train, "r") as f:
	lines = f.readlines()
lines = [
	line.replace(
	   "all_params = [torch.load(i, map_location=torch.device('cpu') ) for i in all_best_model_files]",

	   "all_params = [torch.load(i, map_location=torch.device('cpu') , weights_only=False) for i in all_best_model_files]"
	 )
	for line in lines
]
with open(file_path_train, "w") as f:
  f.writelines(lines)

print("Changes have been applied!")

!pip uninstall nnunet -y
!pip install -r Brats21_KAIST_MRI_Lab/requirements_v2.txt
!cd Brats21_KAIST_MRI_Lab && pip install .

print("NN_UNET Installed!")



In [ ]:
os.environ["nnUNet_raw_data_base"] = "./nnUNet_raw_data_base"
os.environ["nnUNet_preprocessed"] = "./nnUNet_raw_data_base/nnUNet_preprocessed"
os.environ["RESULTS_FOLDER"] = "./nnUNet_raw_data_base/nnUNet_results"

!mkdir -p ./nnUNet_raw_data_base/nnUNet_raw_data/Task501_BraTS2021/imagesTr
!mkdir -p ./nnUNet_raw_data_base/nnUNet_raw_data/Task501_BraTS2021/labelsTr

 ## 2.4 Data Preparation for nnU-Net

 ### 2.4.1 Copy Data

In [ ]:
BRATS_ROOT = Path("./BraTS2021_Training_Data2/BraTS2021_Training_Data2")
TASK_ROOT = Path("./nnUNet_raw_data_base/nnUNet_raw_data/Task501_BraTS2021")
imagesTr = TASK_ROOT / "imagesTr"
labelsTr = TASK_ROOT / "labelsTr"
MODALITY_MAP = {
	"flair": "0000",
	"t1": "0001",
	"t1ce": "0002",
	"t2": "0003"
}
cases = sorted([d for d in BRATS_ROOT.iterdir() if d.is_dir()])
for case_dir in tqdm(cases):
	case_id = case_dir.name
	for mod, suffix in MODALITY_MAP.items():
		src = case_dir / f"{case_id}_{mod}.nii.gz"  
		dst = imagesTr / f"{case_id}_{suffix}.nii.gz"
		if src.exists():
			shutil.copy(src, dst)
	seg_src = case_dir / f"{case_id}_seg.nii.gz"
	seg_dst = labelsTr / f"{case_id}.nii.gz"
	if seg_src.exists():
		shutil.copy(seg_src, seg_dst)


 ### 2.4.2 Remap Labels

In [ ]:
for seg_file in labelsTr.glob("*.nii.gz"):
	img = nib.load(seg_file)
	data = img.get_fdata().astype(np.uint8)
	data[data == 4] = 3
	new_img = nib.Nifti1Image(data, img.affine, img.header)
	nib.save(new_img, seg_file)


 ### 2.4.3 Generate Dataset JSON

In [ ]:
from nnunet.dataset_conversion.utils import generate_dataset_json

In [ ]:
generate_dataset_json(
	str(TASK_ROOT / "dataset.json"),
	str(imagesTr),
	None,
	{0: "FLAIR", 1: "T1", 2: "T1ce", 3: "T2"},
	{0: "background", 1: "necrotic_non_enhancing", 2: "edema", 3: "enhancing_tumor"},
	"BraTS2021"
)

 ## 2.5 Compute Volumes and Save CSV

In [ ]:
LABELS_DIR = Path("./nnUNet_raw_data_base/nnUNet_raw_data/Task501_BraTS2021/labelsTr")
records = []
for seg_file in sorted(LABELS_DIR.glob("*.nii.gz")):
	case_id = seg_file.stem[:-4]  # to remove .nii.gz
	img = nib.load(seg_file)
	data = img.get_fdata()
	spacing = img.header.get_zooms()
	voxel_volume = np.prod(spacing)
	wt_voxels = np.sum(data > 0)
	wt_volume_mm3 = wt_voxels * voxel_volume
	wt_volume_cm3 = wt_volume_mm3 / 1000.0
	records.append({
		"case_id": case_id,
		"wt_voxels": int(wt_voxels),
		"wt_volume_mm3": float(wt_volume_mm3),
		"wt_volume_cm3": float(wt_volume_cm3)
	})
df = pd.DataFrame(records)
df.to_csv("./brats2021_whole_tumor_volumes.csv", index=False)
print(df.head())




 # 3. Model and Result



 ## 3.1 Helper Functions for Image Processing

In [ ]:
def _load_nii(path: str) -> np.ndarray:
	img = nib.load(path)
	return np.asanyarray(img.dataobj).astype(np.float32, copy=False)
def _zscore(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
	m = float(x.mean())
	s = float(x.std())
	return (x - m) / (s + eps)
def _center_crop_or_pad_3d(vol: np.ndarray, target: Tuple[int, int, int]) -> np.ndarray:
	td, th, tw = target
	d, h, w = vol.shape
	out = vol
	# crop
	if d > td: out = out[(d - td)//2 : (d - td)//2 + td, :, :]
	if h > th: out = out[:, (h - th)//2 : (h - th)//2 + th, :]
	if w > tw: out = out[:, :, (w - tw)//2 : (w - tw)//2 + tw]
	# pad
	d2, h2, w2 = out.shape
	pd0, pd1 = max(0, (td - d2)//2), max(0, td - d2 - max(0, (td - d2)//2))
	ph0, ph1 = max(0, (th - h2)//2), max(0, th - h2 - max(0, (th - h2)//2))
	pw0, pw1 = max(0, (tw - w2)//2), max(0, tw - w2 - max(0, (tw - w2)//2))
	if any([pd0, pd1, ph0, ph1, pw0, pw1]):
		out = np.pad(out, ((pd0, pd1), (ph0, ph1), (pw0, pw1)), mode="constant", constant_values=0)
	return out


 ## 3.2 DataLoader

 ### 3.2.1 Define Class Dataset

In [ ]:
@dataclass(frozen=True)
class Sample:
	image_paths: List[str]
	seg_path: Optional[str]
	y: float

class BratsScalarDataset(Dataset):
	def __init__(self, samples: List[Sample], patch_size: Tuple[int, int, int], mask_with_seg: bool = False , y_mean: float = 0.0, y_std: float = 1.0) -> None:
		self.samples = samples
		self.patch_size = patch_size
		self.mask_with_seg = mask_with_seg
		self.y_mean = y_mean
		self.y_std = y_std

	def __len__(self) -> int:
		return len(self.samples)
	def __getitem__(self, idx: int):
		s = self.samples[idx]
		if len(s.image_paths) == 1:
			x = _load_nii(s.image_paths[0])
			if x.ndim == 4:
				x_chw = np.moveaxis(x, -1, 0) if x.shape[-1] <= 8 else x
			else:
				raise ValueError(f"Unexpected shape {x.shape}")
		else:
			vols = [_load_nii(p) for p in s.image_paths]
			x_chw = np.stack(vols, axis=0) # (C, D, H, W)
		seg = None
		if self.mask_with_seg and s.seg_path:
			seg = _load_nii(s.seg_path)
			seg = (seg > 0).astype(np.float32)
			seg = _center_crop_or_pad_3d(seg, self.patch_size)
		c = x_chw.shape[0]
		x_out = np.zeros((c, *self.patch_size), dtype=np.float32)
		for ci in range(c):
			v = _center_crop_or_pad_3d(x_chw[ci], self.patch_size)
			if seg is not None:
				v *= seg
			v = _zscore(v)
			x_out[ci] = v
		
		#Normalaization
		y_raw = float(s.y)
		y_norm = (np.log1p(y_raw) - self.y_mean) / self.y_std if self.y_std != 0 else np.log1p(y_raw)        
		return torch.from_numpy(x_out), torch.tensor([y_norm], dtype=torch.float32)



In [ ]:
def _read_samples_from_csv(csv_path: str, brats_root: str, label_column: str = "wt_volume_cm3") -> List[Sample]:
	df = pd.read_csv(csv_path)
	samples = []
	if "case_id" in df.columns and label_column in df.columns:
		root = Path(brats_root)
		for _, row in df.iterrows():
			case_id = str(row["case_id"])
			case_dir = root / case_id 
			flair_path = str(case_dir / f"{case_id}_flair.nii.gz")
			t1_path = str(case_dir / f"{case_id}_t1.nii.gz")
			t1ce_path = str(case_dir / f"{case_id}_t1ce.nii.gz")
			t2_path = str(case_dir / f"{case_id}_t2.nii.gz")
			seg_path = str(case_dir / f"{case_id}_seg.nii.gz")
			y = float(row[label_column])
			samples.append(Sample(
				image_paths=[flair_path, t1_path, t1ce_path, t2_path],
				seg_path=seg_path,
				y=y
			))
	return samples


 ### 3.2.2 Creat Sample

In [ ]:
csv_path = "./brats2021_whole_tumor_volumes.csv"
brats_root = "./BraTS2021_Training_Data2/BraTS2021_Training_Data2"
label_column = "wt_volume_cm3"
samples = _read_samples_from_csv(csv_path, brats_root, label_column)

In [ ]:
# Stochastically select a subset of cases to reduce computational load
subsample_fraction = 1.0   # 0.1 to 1.0
random.seed(42)  
samples = random.sample(samples, int(len(samples) * subsample_fraction))
print(f"Subsampled to {len(samples)} cases (fraction: {subsample_fraction})")

 ## 3.3 Model Definition

In [ ]:
class NNUNetScalarHead(nn.Module):
	def __init__(self, backbone: nn.Module, n_backbone_channels: int, hidden: int = 128) -> None:
		super().__init__()
		self.backbone = backbone
		self.head = nn.Sequential(
			nn.Flatten(),
			nn.Linear(n_backbone_channels, hidden),
			nn.ReLU(inplace=True),
			nn.Linear(hidden, 1),
		)
	def forward(self, x: torch.Tensor) -> torch.Tensor:
		feat = self.backbone(x)
		if isinstance(feat, (tuple, list)):
			feat = feat[0]
		feat_summed = feat.sum(dim=(-3 , -2 , -1))
		num_voxels = x.shape[-1] * x.shape[-2] * x.shape[-3]   
		feat_sum = feat_summed / num_voxels
		return self.head(feat_sum)
	
def _set_requires_grad(module: nn.Module, requires_grad: bool) -> None:
	for p in module.parameters():
		p.requires_grad = requires_grad


 ## 3.4 Configuration and Execution

 ### 3.4.1 Setup

 **IMPORTANT:**


You can download the model-weights folder from this [link](https://drive.usercontent.google.com/download?id=1HZmWG4j2zQg0vVwBsTrpnuLOmtKCpix2&export=download&authuser=0&confirm=t&uuid=ba826e3c-3631-4115-ab7e-f8d4704f3cc6&at=APcXIO07TXlwWo3JUzTQZrcvawUy%3A1771417962688) , Also, you can request access from [the author](https://mail.google.com/mail/?view=cm&fs=1&to=zahraa.mirzaee.1999@gmail.com) to the folder containing the model weights on Google Drive.


 Before mounting Google Drive in Colab:

 1) In the shared model-weights folder, right-click → Organize

 2) Click "Add shortcut"

 3) Select "My Drive" and confirm

 Then mount Drive below and use the shortcut path inside MyDrive.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# To find the path of Model the file should have this name : (nnUNetTrainerV2BraTSRegions_DA4_BN_BD_largeUnet_Groupnorm__nnUNetPlansv2.1)
# You should add shortcut path on /content/drive/MyDrive/...
!ls /content/drive/MyDrive


In [ ]:
# nnunet_fold_folder = "/content/drive/MyDrive/..." #Add Your Path here
nnunet_fold_folder = "./Task500_BraTS2021/nnUNetTrainerV2BraTSRegions_DA4_BN_BD_largeUnet_Groupnorm__nnUNetPlansv2.1"
csv_path = "./brats2021_whole_tumor_volumes.csv"
brats_root = "./BraTS2021_Training_Data2/BraTS2021_Training_Data2"
label_column = "wt_volume_cm3"
out_ckpt = "scalar_head.pt"
epochs = 100
batch_size = 4
lr = 1e-4
weight_decay = 1e-5
seed = 123
mask_with_seg = True
train_backbone = False 
freeze_seg_outputs = True
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
from nnunet.training.model_restore import load_model_and_checkpoint_files
from nnunet.training.network_training.nnUNetTrainerV2 import nnUNetTrainerV2

In [ ]:
import torch
from torch.serialization import load as original_torch_load

def trusted_load(*args, **kwargs):
	kwargs.setdefault('weights_only', False)
	return original_torch_load(*args, **kwargs)

_prev_load = torch.load
torch.load = trusted_load

try:
	trainer, _ = load_model_and_checkpoint_files(
		nnunet_fold_folder,
		folds=[2], #You Can use the fold : 0,1,2,3,4
		checkpoint_name="model_final_checkpoint"
	)
	print("Load Model Successfully!")

except Exception as e:
	print("Load Error", e)
	raise
finally:
	torch.load = _prev_load


 ### 3.4.2 Load Data and Model

In [ ]:
trainer.initialize(False) # inference mode

patch_size = tuple(trainer.plans['plans_per_stage'][trainer.stage]['patch_size'])

num_input_channels = trainer.plans['num_modalities']


trainer_fold, params_fold = load_model_and_checkpoint_files(
	nnunet_fold_folder,
	folds=[2], #you can use the 0,1,2,3,4 fold
	checkpoint_name="model_final_checkpoint"
)
net_fold = trainer_fold.network
net_fold.do_ds = False
net_fold.eval()
net_fold.to(device)

# Get n_backbone_channels
with torch.no_grad():
	dummy = torch.zeros((1, num_input_channels, *patch_size), device=device)
	out = net_fold(dummy)
	n_backbone_channels = out[0].shape[1] if isinstance(out, tuple) else out.shape[1]

model = NNUNetScalarHead(net_fold, n_backbone_channels).to(device)
if not train_backbone:
	_set_requires_grad(model.backbone, False)
else:
	_set_requires_grad(model.backbone, True)
	if freeze_seg_outputs and hasattr(model.backbone, "seg_outputs"):
		_set_requires_grad(model.backbone.seg_outputs, False)
_set_requires_grad(model.head, True)
	

print({"patch_size": patch_size, "num_input_channels": num_input_channels, "n_backbone_channels": n_backbone_channels})



In [ ]:
y_list = [s.y for s in samples]
# Create bins for stratification (4 quantiles to balance distribution)
bins = pd.qcut(y_list, 4, labels=False, duplicates='drop')

#split
idx = np.arange(len(samples))
train_idx, temp_idx = train_test_split(idx, test_size=0.2, stratify=bins, random_state=123)
temp_bins = [bins[i] for i in temp_idx]  # Update bins for temp
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=temp_bins, random_state=123)

# Samples
tr_samples = [samples[i] for i in train_idx]
val_samples = [samples[i] for i in val_idx]
test_samples = [samples[i] for i in test_idx]

tr_y_log = [np.log1p(s.y) for s in tr_samples]  
y_mean_log = np.mean(tr_y_log)
y_std_log = np.std(tr_y_log)
print(f"Train y_log mean: {y_mean_log}, std: {y_std_log}")

ds_tr = BratsScalarDataset(tr_samples, patch_size, mask_with_seg, y_mean=y_mean_log, y_std=y_std_log)
ds_val = BratsScalarDataset(val_samples, patch_size, mask_with_seg, y_mean=y_mean_log, y_std=y_std_log)
ds_test = BratsScalarDataset(test_samples, patch_size, mask_with_seg, y_mean=y_mean_log, y_std=y_std_log)

dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
dl_val = DataLoader(ds_val, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
dl_test = DataLoader(ds_test, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)


 ### 3.4.3 Plots and Evaluation Functions

In [ ]:

def plot_predictions(true_y, pred_y, title='Predicted vs Actual Volume', save_path="./Plot/plot_predictions"):
	plt.figure(figsize=(8, 6))
	sns.scatterplot(x=true_y, y=pred_y, alpha=0.7)
	plt.plot([min(true_y), max(true_y)], [min(true_y), max(true_y)], 'r--')  
	plt.xlabel('True Volume (cm³)')
	plt.ylabel('Predicted Volume (cm³)')
	plt.title(title)
	if save_path:
		plt.savefig(save_path)
	plt.show()

def plot_error_histogram(errors, title='Histogram of Prediction Errors', save_path="./Plot/plot_error_histogram"):
	plt.figure(figsize=(8, 6))
	sns.histplot(errors, bins=20, kde=True)
	plt.xlabel('Error (cm³)')
	plt.title(title)
	if save_path:
		plt.savefig(save_path)
	plt.show()

def plot_bland_altman(true_y, pred_y, title='Bland-Altman Plot', save_path="./Plot/plot_bland_altman"):
	mean = (np.array(true_y) + np.array(pred_y)) / 2
	diff = np.array(true_y) - np.array(pred_y)
	plt.figure(figsize=(8, 6))
	sns.scatterplot(x=mean, y=diff, alpha=0.7)
	plt.axhline(np.mean(diff), color='r', linestyle='--')  
	plt.axhline(np.mean(diff) + 1.96 * np.std(diff), color='g', linestyle='--')  
	plt.axhline(np.mean(diff) - 1.96 * np.std(diff), color='g', linestyle='--')  
	plt.xlabel('Mean Volume (cm³)')
	plt.ylabel('Difference (True - Pred)')
	plt.title(title)
	if save_path:
		plt.savefig(save_path)
	plt.show()


 ### 3.4.4 Metrics

In [ ]:
def mean_absolute_percentage_error(true_y, pred_y, epsilon=1e-6):
	true_y = np.array(true_y)
	pred_y = np.array(pred_y)
	return np.mean(np.abs((true_y - pred_y) / (true_y + epsilon))) * 100  

def concordance_correlation_coefficient(true_y, pred_y):
	true_y = np.array(true_y)
	pred_y = np.array(pred_y)
	cor = np.corrcoef(true_y, pred_y)[0][1]
	mean_true = np.mean(true_y)
	mean_pred = np.mean(pred_y)
	var_true = np.var(true_y)
	var_pred = np.var(pred_y)
	sd_true = np.std(true_y)
	sd_pred = np.std(pred_y)
	numerator = 2 * cor * sd_true * sd_pred
	denominator = var_true + var_pred + (mean_true - mean_pred)**2
	return numerator / denominator if denominator != 0 else 0

def evaluate_metrics(true_y, pred_y):
	mae = mean_absolute_error(true_y, pred_y)
	mape = mean_absolute_percentage_error(true_y, pred_y)
	r2 = r2_score(true_y, pred_y)
	ccc = concordance_correlation_coefficient(true_y, pred_y)
	print(f"MAE: {mae:.4f} cm³")
	print(f"MAPE: {mape:.2f}%")
	print(f"R²: {r2:.4f}")
	print(f"CCC: {ccc:.4f}")
	return {'MAE': mae, 'MAPE': mape, 'R2': r2, 'CCC': ccc}


 ## 3.5 Training Loop

In [ ]:
optim = torch.optim.AdamW(
	filter(lambda param: param.requires_grad, model.parameters()),
	lr=lr, weight_decay=weight_decay
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, patience=5, factor=0.5)


In [ ]:
loss_fn = nn.HuberLoss(delta=1.0)

best_val_mae = float("inf")
for epoch in range(epochs):
	model.train()
	if not train_backbone:
		model.backbone.eval()

	tr_losses = []

	for xb, yb in tqdm(dl_tr, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
		xb, yb = xb.to(device), yb.to(device)
		optim.zero_grad()

		pred_norm = model(xb)
		loss = loss_fn(pred_norm, yb)
		loss.backward()
		optim.step()

		tr_losses.append(loss.item())


	model.eval()
	val_losses = []
	val_true = []
	val_pred = []

	with torch.no_grad():
		for xb, yb in tqdm(dl_val, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
			
			yb = yb.to(device)
			xb= xb.to(device)
	
			
			pred_norm = model(xb)
			pred = np.expm1(pred_norm.item() * y_std_log + y_mean_log)  
			true_y = np.expm1(yb.item() * y_std_log + y_mean_log)
			
			val_true.append(true_y)
			val_pred.append(pred)
			loss = loss_fn(pred_norm, yb)  
			val_losses.append(loss.item())

	tr_loss = np.mean(tr_losses)
	val_loss = np.mean(val_losses)
	metrics = evaluate_metrics(val_true, val_pred)
	
	val_mae = mean_absolute_error(val_true, val_pred)
	scheduler.step(val_mae)
	print(f"epoch={epoch+1:03d} train_loss={tr_loss:.6f} val_loss={val_loss:.6f}")

	if val_mae < best_val_mae:
		best_val_mae = val_mae
		best_epoch = epoch + 1

		torch.save(model.head.state_dict(),f"best_scalar_head_fold.pth")
		torch.save(model.state_dict(), f"full_model.pth")

		print(f"→ Best model saved at epoch {best_epoch} (Val MAE: {best_val_mae:.4f} cm³)")

	
	




 # 4. Test Model

 ## 4.1 Test

In [ ]:


best_fold_checkpoint_paths = [
	"./full_model0_2.pth", # If you have some of the folds, you shoud add path .pth file here
 
]
loss_fn = nn.HuberLoss(delta=1.0)

y_mean_log = np.mean(tr_y_log)
y_std_log = np.std(tr_y_log)
patch_size = tuple(trainer.plans['plans_per_stage'][trainer.stage]['patch_size'])

models = []

for fold_idx, head_ckpt_path in enumerate(best_fold_checkpoint_paths):
	
	trainer_fold, _ = load_model_and_checkpoint_files(
		nnunet_fold_folder,
		folds=[2],        # FoldNumber , If you in best_fold_checkpoint_paths have some of the fold you should add the number of fold based the weight-fold from nn_unet used add here[0,1,2,3,4] or if the path are sorted 0 to 4, you can use the fold_idx.
		checkpoint_name="model_final_checkpoint"
	)
	
	net_fold = trainer_fold.network
	net_fold.do_ds = False
	net_fold.eval()
	net_fold.to(device)
	
	if fold_idx == 0:
		with torch.no_grad():
			dummy = torch.zeros((1, num_input_channels, *patch_size), device=device)
			out = net_fold(dummy)
			n_backbone_channels = out[0].shape[1] if isinstance(out, tuple) else out.shape[1]
	
	model_fold = NNUNetScalarHead(net_fold, n_backbone_channels).to(device)
	
	state_dict = torch.load(head_ckpt_path, map_location=device, weights_only=False)
	model_fold.load_state_dict(state_dict)
	
	model_fold.eval()
	for param in model_fold.parameters():
		param.requires_grad = False
	
	models.append(model_fold)

print(f"→ {len(models)} Load Models Success!")

for idx , model in enumerate(models):
	model.eval()
	true_y_list = []
	pred_y_list = []
	with torch.no_grad():
		for xb, yb in dl_test:  
			yb = yb.to(device)
			pred_norms = []
			for m in models:
				pred_norm = m(xb.to(device))
				pred_norms.append(pred_norm)
			pred = np.expm1(pred_norm.item() * y_std_log + y_mean_log)  
			true_y = np.expm1(yb.item() * y_std_log + y_mean_log)

			true_y_list.append(true_y)
			pred_y_list.append(pred)
			print(f"True volume: {true_y:.2f} cm³, Predicted: {pred:.2f} cm³")

	metrics = evaluate_metrics(true_y_list, pred_y_list)
    
	#Plot
	errors = np.array(true_y_list) - np.array(pred_y_list)
	plot_predictions(true_y_list, pred_y_list, title=f'Test Set: Predicted vs Actual{idx}')
	plot_error_histogram(errors, title=f'Test Set: Error Histogram{idx}')
	plot_bland_altman(true_y_list, pred_y_list, title=f'Test Set: Bland-Altman{idx}')



